# Chapter 2: LLM API Basics

In [2]:
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from litellm import completion
from pydantic import BaseModel, Field

load_dotenv(find_dotenv())

True

## Listing 2.2: Basic OpenAI API Call

In [2]:
client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)   

In [3]:
print(response.choices[0].message.content)

The capital of France is Paris.


### Experiment: API can write several independent answers to the same prompt in one request

In [4]:
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Suggest a name for a hotel room-service bot."},
    ],
    n=3
)

for i, choice in enumerate(response.choices):
    print(i, choice.finish_reason, choice.message.content)
    
print(response.usage)

0 stop Here are some name ideas, grouped by tone. Short tagline included so you can picture the personality.

Luxury / Elegant
- Orchid — refined, calming service
- Velvet — smooth, discreet help
- Lumo — soft, modern luxury
- Tranquil — restful, concierge-like

Friendly / Warm
- SuiteMate — your in-room companion
- PillowPal — cozy, approachable assistance
- Hostly — friendly, helpful host
- ParcelPal — practical and personable

Butler / Concierge-style
- RoomButler — classic, reliable service
- ConciergeOne — professional, single-touch help
- Minerva — wise, attentive concierge
- KeyMate — personal, on-demand assistant

Tech-forward / Modern
- EchoServe — responsive, voice-ready bot
- ServeSense — smart, anticipatory service
- Quiki — snappy, efficient requests
- Halo — simple, clean interface

Playful / Charming
- NiteOwl — night-service specialist
- Turndown — charmingly on-brand for hotels
- NimbleNook — quick, clever helper
- BellaServe — friendly with a touch of warmth

Top 3 qu

## Listing 2.5: Demonstrating API Statelessness

In [5]:
# First call
response1 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is Sharon."}]
)

#Second call
response2 = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is my name?"}]
)

In [6]:
print(response2.choices[0].message.content)

I don't know — I don't have access to your personal data unless you tell me. What would you like me to call you? (You can reply, for example, "Call me Alex" or "My name is Maria".)


## Listing 2.6: Managing conversation history

In [7]:
messages = []

#First exchange
messages.append({"role": "user", "content": "My name is Sharon!"})
response1 = completion(model="gpt-5-mini", messages=messages) 

In [8]:
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})

In [9]:
print(assistant_message1)

Hi Sharon — nice to meet you! How can I help you today?


In [10]:
#Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model="gpt-5-mini", messages=messages)
assistant_message2 = response2.choices[0].message.content

In [11]:
print(assistant_message2)

Your name is Sharon.


In [12]:
print(response2.usage.prompt_tokens)

41


In [13]:
print(response1.usage.prompt_tokens)

11


## Listing 2.7: Structured output with Pydantic

In [14]:
class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

In [15]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith, my email is John@example.com, and my phone is 555-1234"}],
    response_format=ExtractedInfo
)

In [16]:
result = response.choices[0].message.content

In [17]:
print(type(result))

<class 'str'>


In [18]:
info = ExtractedInfo.model_validate_json(result)

In [19]:
info.email

'John@example.com'

In [20]:
info.phone

'555-1234'

### Experiment: What happens when the one field in the schema is missing?

In [21]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith, my email is John@example.com"}],
    response_format=ExtractedInfo
)

In [22]:
result = response.choices[0].message.content

In [23]:
info = ExtractedInfo.model_validate_json(result)

In [24]:
info.phone

In [25]:
print(result)

{"name":"John Smith","email":"John@example.com","phone":null}


In [26]:
response = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is John Smith"}],
    response_format=ExtractedInfo
)

In [27]:
result = response.choices[0].message.content

In [28]:
info = ExtractedInfo.model_validate_json(result)

In [29]:
info.email

''

In [30]:
print(type(info))

<class '__main__.ExtractedInfo'>


In [31]:
info.phone

### Learning: Pydantic only errors when the key is missing or the value is the wrong type.

In [33]:
from pydantic import ValidationError
try:
    ExtractedInfo.model_validate_json('{"name": "A"}')
except ValidationError as e:
    print(e)

1 validation error for ExtractedInfo
email
  Field required [type=missing, input_value={'name': 'A'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [34]:
try:
    ExtractedInfo.model_validate_json('{"name": "A", "email": 5}')
except ValidationError as e:
    print(e)

1 validation error for ExtractedInfo
email
  Input should be a valid string [type=string_type, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type


### Experiment: Use Field(...) to attach extra rules to a field

In [35]:
class ExtractedInfo(BaseModel):
    name: str
    email: str = Field(min_length=1)
    phone: str | None = None

In [36]:
try:
    ExtractedInfo.model_validate_json('{"name": "A", "email": ""}')
except ValidationError as e:
    print(e)

1 validation error for ExtractedInfo
email
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short


### Experiment: Why making a field optional is usually the better design when the data might genuinely be missing?

In [37]:
class ExtractedInfo(BaseModel):
    name: str
    email: str | None = None
    phone: str | None = None

In [38]:
for i in range(5):
    response = completion(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": "My name is John Smith"}],
        response_format=ExtractedInfo
    )
    info = ExtractedInfo.model_validate_json(response.choices[0].message.content)
    print(i, info.email)

0 None
1 None
2 None
3 None
4 None


In [39]:
for i in range(5):
    response = completion(
        model="gpt-5-mini",
        messages=[{"role": "system", "content": "Only extract values stated in the text. If a field isn't mentioned, use null. Never guess."},{"role": "user", "content": "My name is John Smith"}],
        response_format=ExtractedInfo
    )
    info = ExtractedInfo.model_validate_json(response.choices[0].message.content)
    print(i, info.email)

0 None
1 None
2 None
3 None
4 None


1. The schema change mattered: required str gave '', and optional gave None every time. Just giving the model a legal “missing” answer changed its behavior.
2. The instruction didn’t matter here, because the case was too easy. Prompts make a difference on tempting inputs, like “email me at my usual Gmail” or “it’s my name at the company domain”. There, a guess looks reasonable.
3. 5/5 isn’t proof: a model that fails 1 time in 10 would still often pass 5 in a row. Real evals use dozens to hundreds of cases, which is why GAIA uses 20 problems even in the quick version. You’ll see this again in Ch10.

## Listing 2.8: Asynchronous LLM Calls

In [3]:
import asyncio
from litellm import acompletion

In [4]:
async def get_response(prompt: str) -> str:
    response = await acompletion(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": "prompt"}]
    )
    return response.choices[0].message.content

### Demo: How async and await work

In [3]:
def normal():
    return "Hi"

async def special():
    return "Special Hi"

In [4]:
normal()

'Hi'

In [5]:
special()

<coroutine object special at 0x111827ed0>

In [6]:
await special()

'Special Hi'

In [7]:
special()
print("done")

done


/var/folders/lq/dxlhq_n53t55bc4vg11nd5rc0000gn/T/ipykernel_19439/70695771.py:1: RuntimeWarning: coroutine 'special' was never awaited
  special()


If you forget an await in front of acompletion(...), the API call never happens and you get no error, only this warning. If the warning is missing as well, the code just silently does nothing.

#### Whenever a cell’s last line produces a value, Jupyter quietly saves it in a few places

In [8]:
special()

<coroutine object special at 0x117f70670>

In [9]:
Out[8]

<coroutine object special at 0x117f70670>

In [10]:
_8

<coroutine object special at 0x117f70670>

In [13]:
__

<coroutine object special at 0x117f70670>

In [14]:
await Out[8]

'Special Hi'

In [15]:
await Out[8]

RuntimeError: cannot reuse already awaited coroutine

an async function only runs when you await it. If you forget the await, nothing happens, and you may or may not get a warning. 

### Continuing Listing 2.8

In [5]:
prompts = [
            "What is 2 + 2?",
            "What is the capital of Japan?",
            "Who wrote Romeo and Juliet?"
]

In [6]:
# Execute all requests concurrently
tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

Why it didn't write ```tasks = [await get_response(p) for p in prompts]```?

gather takes all 3 tasks and starts at once, then waits until all 3 are done. The * unpacks the list: gather(*[a, b, c]) is the same as gather(a, b, c), because gather wants separate tasks, not a list. results comes back in the same order as prompts.

Here the await sits inside the loop, so for each prompt it writes the task, hands it over, and waits until that task is done before writing the next one. That’s sequential: three waits back to back. (Also, tasks would hold the 3 answers, not tasks, so it would be misnamed.)

The rule: await means “stop here until this one finishes”. To run things at the same time, first create all the coroutines without awaiting them, then await them together with gather.

### Why gather, and not await inside the loop?

**Concurrent version (the book's):** `results = await asyncio.gather(*tasks)`
[1–2 sentences: what the list comprehension creates, what gather does, what * does]

**Sequential version:** `[await get_response(p) for p in prompts]`
[1 sentence: why this waits for each call before starting the next]

**Rule:** [your one-liner about await and gather]

**Note: concurrent ≠ parallel.** [1 sentence: one worker overlapping the *waiting*, not three workers doing the work at once]

In [7]:
for prompt, result in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result}\n")

Q: What is 2 + 2?
A: Do you mean an example prompt (for writing, image generation, coding, a system prompt for an LLM, etc.) — or something else? I can create a prompt tailored to your goal. Below are quick examples across common uses; tell me which one you want expanded or customized.

Examples

1) ChatGPT / conversation assistant (short)
- “Act as a friendly travel planner. Ask about my budget, interests, and travel dates, then propose a 5-day itinerary for Kyoto with daily plans, transport tips, and budget estimates.”

2) ChatGPT / detailed system-style
- “You are an expert product manager. When I give you an idea, evaluate market fit, target users, KPIs, feature roadmap (MVP → v1.0 → v2.0), key risks, and a rough 6‑month timeline with milestones.”

3) Creative writing / story starter
- “Write a 1200-word first-person story where a retired lighthouse keeper discovers a message in a bottle that turns out to be decades old and changes everything he thought he knew about his past.”

4)

## Listing 2.9: Rate-limited Concurrent Requests

### Free Demo: How Semaphore works

In [8]:
import time

demo_sem = asyncio.Semaphore(2)

async def worker(name):
    async with demo_sem:
        print(time.strftime("%X"), name, "IN")
        await asyncio.sleep(1)
        print(time.strftime("%X"), name, "OUT")

await asyncio.gather(*[worker(f"w{i}") for i in range(5)])

11:37:47 w0 IN
11:37:47 w1 IN
11:37:48 w0 OUT
11:37:48 w1 OUT
11:37:48 w2 IN
11:37:48 w3 IN
11:37:49 w2 OUT
11:37:49 w3 OUT
11:37:49 w4 IN
11:37:50 w4 OUT


[None, None, None, None, None]

In [9]:
#Limit to 10 concurrent requests
semaphore = asyncio.Semaphore(10)

In [10]:
async def call_llm(prompt: str) -> str:
    """LLM Call with Rate Limiting and Automatic Retry"""
    async with semaphore:
        response = await acompletion(
            model="gpt-5-mini",
            messages=[{"role": "user", "content": prompt}],
            num_retries=3 # automatic retry with exponential backoff
        )
        return response.choices[0].message.content

In [11]:
prompts = [f"What is {i} + {i}?" for i in range(5)]
tasks = [call_llm(p) for p in prompts]
results = await asyncio.gather(*tasks, return_exceptions=True)